In [ ]:
##############################################################################
# BDD100K Semantic Segmentation — DeepLabV3 (ResNet-50)
# Single-cell Kaggle notebook: train → validate → export TorchScript
##############################################################################

import os, glob, csv, warnings, numpy as np, pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import random
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torchvision.models.segmentation as seg_models
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ── 0. Device ───────────────────────────────────────────────────────────────────────
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
print(f"Using device: {device}  (CUDA devices = {torch.cuda.device_count()})")

# ── 1. Dataset paths & class mapping (BDD100K) ──────────────────────────────────────
# Change ROOT if your dataset is mounted elsewhere (e.g. /kaggle/input/...)
ROOT = os.environ.get("BDD_ROOT", "/kaggle/input/bdd100k")
TRAIN_IMG = os.path.join(ROOT, "images", "train")
VAL_IMG   = os.path.join(ROOT, "images", "val")
TRAIN_LBL = os.path.join(ROOT, "labels", "train")
VAL_LBL   = os.path.join(ROOT, "labels", "val")

# Prefer using an external Carla-style colour CSV if provided (label_colors.csv)
CLASS_CSV = "label_colors.csv"

# Attempt to load class colours from CSV, fallback to class_dict.csv or build from masks
if os.path.exists(CLASS_CSV):
    class_df = pd.read_csv(CLASS_CSV)
    class_names = class_df["name"].tolist()
    class_rgb   = class_df[["r", "g", "b"]].values.astype(np.uint8)
elif os.path.exists("class_dict.csv"):
    class_df = pd.read_csv("class_dict.csv")
    class_names = class_df["name"].tolist()
    class_rgb   = class_df[["r", "g", "b"]].values.astype(np.uint8)
else:
    # Build a colour-to-class mapping by scanning a sample of label images (useful when no CSV available)
    sample_mask_files = sorted(glob.glob(os.path.join(TRAIN_LBL, "*.png")))[:200]
    colours = {}
    for fp in sample_mask_files:
        arr = np.array(Image.open(fp).convert("RGB"))
        uniq = np.unique(arr.reshape(-1, 3), axis=0)
        for c in uniq:
            colours[tuple(c)] = colours.get(tuple(c), 0) + 1
    unique_colors = sorted(colours.keys(), key=lambda c: -colours[c])
    class_names = [f"class_{i}" for i in range(len(unique_colors))]
    if len(unique_colors) == 0:
        raise RuntimeError("No label colours found in TRAIN_LBL — check your dataset paths")
    class_rgb = np.array(unique_colors, dtype=np.uint8)
    class_df = pd.DataFrame({
        "id": list(range(len(class_names))),
        "name": class_names,
        "r": class_rgb[:, 0],
        "g": class_rgb[:, 1],
        "b": class_rgb[:, 2],
    })
    # Save CSV so future runs reuse the mapping
    class_df.to_csv(CLASS_CSV, index=False)

NUM_CLASSES = len(class_names)
print(f"Classes ({NUM_CLASSES}): {class_names[:10]}{'...' if NUM_CLASSES>10 else ''}")

# Build a fast lookup: (r, g, b) → class index
rgb_to_idx = {}
for idx, rgb in enumerate(class_rgb):
    rgb_to_idx[tuple(rgb)] = idx

# ── 2. Dataset + augmentations ─────────────────────────────────────────────────────
IMG_H, IMG_W = 360, 480
img_normalize = T.Normalize(mean=[0.485, 0.456, 0.406],
                            std=[0.229, 0.224, 0.225])
mask_resize = T.Resize((IMG_H, IMG_W), interpolation=T.InterpolationMode.NEAREST)

def mask_to_class_idx(mask_pil):
    """Convert an RGB mask PIL image to a 2-D tensor of class indices."""
    mask_np = np.array(mask_resize(mask_pil))                       # (H, W, 3)
    h, w, _ = mask_np.shape
    idx_map = np.zeros((h, w), dtype=np.int64)
    flat = mask_np.reshape(-1, 3)
    for i, px in enumerate(flat):
        idx_map.flat[i] = rgb_to_idx.get(tuple(px), 0)
    return torch.from_numpy(idx_map)

class BDDDataset(Dataset):
    def __init__(self, img_dir, lbl_dir, augment=False):
        self.imgs = sorted(glob.glob(os.path.join(img_dir, "*.png")))
        self.lbls = sorted(glob.glob(os.path.join(lbl_dir, "*.png")))
        # If no labels are present (test split), create placeholder masks
        if len(self.lbls) == 0:
            self.lbls = [None] * len(self.imgs)
        else:
            assert len(self.imgs) == len(self.lbls), \
,
Mismatch: {len(self.imgs)} imgs vs {len(self.lbls)} labels"
        self.augment = augment

    def __len__(self):
        return len(self.imgs)

    def _apply_augmentation(self, img, lbl):
        if random.random() > 0.5:
            img = TF.hflip(img)
            if lbl is not None: lbl = TF.hflip(lbl)
        if random.random() > 0.5:
            angle = random.uniform(-10, 10)
            img = TF.rotate(img, angle, interpolation=T.InterpolationMode.BILINEAR)
            if lbl is not None: lbl = TF.rotate(lbl, angle, interpolation=T.InterpolationMode.NEAREST)
        if random.random() > 0.5:
            img = TF.adjust_brightness(img, random.uniform(0.8, 1.2))
        if random.random() > 0.5:
            img = TF.adjust_contrast(img, random.uniform(0.8, 1.2))
        return img, lbl

    def __getitem__(self, idx):
        img = Image.open(self.imgs[idx]).convert("RGB")
        lbl_path = self.lbls[idx]
        lbl = Image.open(lbl_path).convert("RGB") if lbl_path is not None else None

        img = TF.resize(img, [IMG_H, IMG_W])

        if self.augment and lbl is not None:
            lbl_resized = mask_resize(lbl)
            img, lbl_resized = self._apply_augmentation(img, lbl_resized)
            mask_np = np.array(lbl_resized)
            h, w, _ = mask_np.shape
            idx_map = np.zeros((h, w), dtype=np.int64)
            flat = mask_np.reshape(-1, 3)
            for i, px in enumerate(flat):
                idx_map.flat[i] = rgb_to_idx.get(tuple(px), 0)
            lbl_tensor = torch.from_numpy(idx_map)
        else:
            lbl_tensor = mask_to_class_idx(lbl) if lbl is not None else torch.zeros((IMG_H, IMG_W), dtype=torch.int64)

        img = TF.to_tensor(img)
        img = img_normalize(img)
        return img, lbl_tensor

# ── 3. Dataloaders ──────────────────────────────────────────────────────────────────
BATCH = 8
NUM_WORKERS = 2
train_ds = BDDDataset(TRAIN_IMG, TRAIN_LBL, augment=True)
val_ds   = BDDDataset(VAL_IMG,   VAL_LBL,   augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Splits — train: {len(train_ds)}, val: {len(val_ds)}")

# ── 3b. Compute class weights (sampled for speed) ──────────────────────────────────
print("Computing class frequencies for weighted loss (sample)...")
class_pixel_count = np.zeros(NUM_CLASSES, dtype=np.int64)
for i, (_, mask) in enumerate(tqdm(train_ds, desc="Counting pixels", leave=False)):
    vals, counts = torch.unique(mask, return_counts=True)
    for v, c in zip(vals.numpy(), counts.numpy()):
        class_pixel_count[v] += c
    if i >= 200:
        break

freq = class_pixel_count / class_pixel_count.sum()
weights = 1.0 / (freq + 1e-6)
weights = weights / weights.sum() * NUM_CLASSES
class_weights = torch.FloatTensor(weights).to(device)
print(f"Class weights (min={class_weights.min():.3f}, max={class_weights.max():.3f})")

# ── 4. Model ────────────────────────────────────────────────────────────────────────
model = seg_models.deeplabv3_resnet50(weights="DEFAULT")
model.classifier[4] = nn.Conv2d(256, NUM_CLASSES, kernel_size=1)
try:
    model.aux_classifier[4] = nn.Conv2d(256, NUM_CLASSES, kernel_size=1)
except Exception:
    pass

# Use DataParallel if multiple GPUs are available
if torch.cuda.device_count() > 1:
    print(f"Using DataParallel with {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

model = model.to(device)

# ── 5. Loss, optimizer & scheduler ──────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(weight=class_weights)
LR = 3e-5
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=4, verbose=True)

# ── 6. Metric helpers ───────────────────────────────────────────────────────────────
def compute_metrics(preds, targets, num_classes):
    preds = preds.flatten()
    targets = targets.flatten()
    correct = (preds == targets).sum().item()
    total = targets.numel()
    pixel_acc = correct / total if total > 0 else 0.0
    ious = []
    for c in range(num_classes):
        pred_c = (preds == c)
        targ_c = (targets == c)
        intersection = (pred_c & targ_c).sum().item()
        union = (pred_c | targ_c).sum().item()
        if union > 0:
            ious.append(intersection / union)
    miou = np.mean(ious) if ious else 0.0
    return pixel_acc, miou, miou

# ── 7. Training loop with early stopping (30 epochs) ───────────────────────────────
NUM_EPOCHS = 30
PATIENCE = 7
best_val_loss = float("inf")
patience_counter = 0
best_model_state = None
history = {"train_loss": [], "val_loss": [], "miou": [], "pixel_acc": [], "jaccard": [], "lr": []}

for epoch in range(1, NUM_EPOCHS + 1):
    current_lr = optimizer.param_groups[0]["lr"]
    history["lr"].append(current_lr)

    # ---- train ----
    model.train()
    running_loss = 0.0
    num_train_samples = 0
    for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [train]", leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        out = model(imgs)["out"]                       # (B, C, H, W)
        loss = criterion(out, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        num_train_samples += imgs.size(0)

    train_loss = running_loss / num_train_samples

    # ---- validate ----
    model.eval()
    val_running_loss = 0.0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [val]", leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            out = model(imgs)["out"]
            loss = criterion(out, masks)
            val_running_loss += loss.item() * imgs.size(0)
            preds = out.argmax(1)
            all_preds.append(preds.cpu())
            all_targets.append(masks.cpu())

    val_loss = val_running_loss / len(val_ds)
    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    pixel_acc, miou, jaccard = compute_metrics(all_preds, all_targets, NUM_CLASSES)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["miou"].append(miou)
    history["pixel_acc"].append(pixel_acc)
    history["jaccard"].append(jaccard)

    # Step LR scheduler
    scheduler.step(val_loss)

    print(f"Epoch {epoch:>2}/{NUM_EPOCHS}  lr={current_lr:.2e}  "
          f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
          f"mIoU={miou:.4f}  pxAcc={pixel_acc:.4f}")

    # ---- early stopping ----
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save model weights (handle DataParallel)
        state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        best_model_state = {k: v.cpu().clone() for k, v in state.items()}
        print(f"  ↳ New best val_loss — checkpoint saved")
    else:
        patience_counter += 1
        print(f"  ↳ No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print(f\
)
            break

# Restore best model weights
if best_model_state is not None:
    model_to_load = model.module if isinstance(model, nn.DataParallel) else model
    model_to_load.load_state_dict(best_model_state)
    model = model.to(device)
    print("Restored best model weights from checkpoint.")

actual_epochs = len(history["train_loss"])

# ── 8. Plots ────────────────────────────────────────────────────────────────────────
epochs_range = range(1, actual_epochs + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs_range, history["train_loss"], label="Train Loss")
axes[0].plot(epochs_range, history["val_loss"],   label="Val Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Training vs Validation Loss")
axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs_range, history["miou"], label="mIoU", color="green")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("mIoU")
axes[1].set_title("Mean IoU vs Epoch")
axes[1].legend(); axes[1].grid(True)

axes[2].plot(epochs_range, history["lr"], label="Learning Rate", color="orange")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("LR")
axes[2].set_title("Learning Rate Schedule")
axes[2].set_yscale("log")
axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.savefig("/kaggle/working/training_curve.png", dpi=150)
plt.show()

# ── 9. Visualisation / Predictions ────────────────────────────────────────────────
inv_normalize = T.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225],
)

cmap_colors = class_rgb.astype(np.float32) / 255.0
seg_cmap = mcolors.ListedColormap(cmap_colors)
seg_norm = mcolors.BoundaryNorm(np.arange(NUM_CLASSES + 1) - 0.5, NUM_CLASSES)

NUM_VIS = 6
vis_indices = np.random.choice(len(val_ds), size=min(NUM_VIS, len(val_ds)), replace=False)

fig, axes = plt.subplots(len(vis_indices), 3, figsize=(15, 4 * len(vis_indices)))
if len(vis_indices) == 1:
    axes = axes[np.newaxis, :]

model.eval()
with torch.no_grad():
    for row, idx in enumerate(vis_indices):
        img_t, gt_mask = val_ds[idx]
        img_vis = inv_normalize(img_t).clamp(0, 1).permute(1, 2, 0).numpy()
        out = model(img_t.unsqueeze(0).to(device))["out"]
        pred_mask = out.argmax(1).squeeze().cpu().numpy()

        axes[row, 0].imshow(img_vis)
        axes[row, 0].set_title("Image")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(gt_mask.numpy(), cmap=seg_cmap, norm=seg_norm)
        axes[row, 1].set_title("Ground Truth")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(pred_mask, cmap=seg_cmap, norm=seg_norm)
        axes[row, 2].set_title("Prediction")
        axes[row, 2].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/test_prediction.png", dpi=150)
plt.show()

# ── 10. Export to TorchScript ───────────────────────────────────────────────────────
EXPORT_PATH = "/kaggle/working/segmentation_model_bdd100k.pt"
script_model = (model.module if isinstance(model, nn.DataParallel) else model).cpu().eval()
scripted = torch.jit.script(script_model)
scripted.save(EXPORT_PATH)
print(f\
)

# ── 11. Final summary ───────────────────────────────────────────────────────────────
best_miou = max(history['miou']) if history['miou'] else 0.0
print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print(f"  Classes         : {NUM_CLASSES}")
print(f"  Epochs ran      : {actual_epochs} / {NUM_EPOCHS}")
print(f"  Best val loss   : {best_val_loss:.4f}")
print(f"  Best val mIoU   : {best_miou:.4f}")
print(f"  TorchScript     : {EXPORT_PATH}")
print("=" * 60)
